# TP4 : Dataset, DataLoader et entraînement d'un CNN

Objectif : partir des photos CIFAR-10 rangées en fichiers, comme dans un vrai projet, écrire votre propre Dataset, le servir par minibatchs avec des DataLoader, puis définir un CNN et l'entraîner. Ce pipeline sert de point de départ au TP5.

L'énoncé complet, avec les critères de réussite, est dans `enonce.md`.

*Complétez les cellules marquées `# TODO`, dans l'ordre.*

## Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

## Préparation (fournie)

**Données** : CIFAR-10 (Krizhevsky, Nair et Hinton, 2009), fourni par torchvision : des photos couleur de 32 x 32 pixels, en 10 classes (avion, automobile, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion). Le site du jeu ne donne pas de licence ; il demande de citer le rapport *Learning Multiple Layers of Features from Tiny Images* (Krizhevsky, 2009). Tous les TP utilisent les mêmes images : 5000 d'entraînement et 1000 de validation, tirées de l'entraînement de CIFAR-10, et 1000 de test.

Les images sont rangées comme dans un projet : un dossier `cifar10_fichiers/images/` avec une image PNG par fichier, et un fichier CSV par partie (`train.csv`, `val.csv`, `test.csv`) qui donne la classe de chaque image, sous forme de numéro.

In [ ]:
def preparer_archive_cifar10(racine="data"):
    """Place les fichiers de CIFAR-10 dans racine/cifar-10-batches-py sans passer par le serveur d'origine, lent.

    Dans l'ordre : fichiers déjà décompressés ; archive déposée dans la session (cifar-10-python.tar.gz ou
    cifar-10-batches-py.tar.gz, dans le dossier du notebook ou dans racine) ; archive d'origine copiée sur le
    Google Drive de la formation. Si rien n'aboutit, torchvision télécharge ensuite depuis le serveur d'origine ;
    dans tous les cas, il vérifie l'intégrité des fichiers.
    """
    import tarfile
    import urllib.request
    from pathlib import Path

    if Path(racine, "cifar-10-batches-py").exists():
        return
    noms = ("cifar-10-python.tar.gz", "cifar-10-batches-py.tar.gz")
    deposees = [Path(dossier, nom) for dossier in (".", racine) for nom in noms if Path(dossier, nom).exists()]
    if deposees:
        archive = deposees[0]
        print(f"Archive trouvée : {archive}")
    else:
        archive = Path(racine, "cifar-10-python.tar.gz")
        archive.parent.mkdir(parents=True, exist_ok=True)
        url = ("https://drive.usercontent.google.com/download?id=1kyq-1VhOPNr2Bo0w5DiLYuNghOdOWl0a"
               "&export=download&confirm=t")
        print("Téléchargement de CIFAR-10 depuis le Google Drive de la formation (170 Mo)...")
        try:
            urllib.request.urlretrieve(url, archive)
        except OSError as erreur:
            print(f"Google Drive indisponible ({erreur}) : téléchargement depuis le serveur d'origine.")
            return
    try:
        with tarfile.open(archive) as contenu:
            contenu.extractall(racine, filter="data")
    except tarfile.TarError:
        print(f"{archive} n'est pas une archive valide : téléchargement depuis le serveur d'origine.")


def charger_cifar10_fichiers(dossier="cifar10_fichiers", n_train=5000, n_val=1000, n_test=1000, seed=0):
    """Les images de charger_cifar10, rangées comme dans un projet : un dossier d'images et un CSV par partie.

    dossier/images/<partie>_<numero>.png, et dossier/<partie>.csv avec une ligne d'en-tête « fichier,classe »
    puis une ligne par image. La classe est un numéro : l'ordre alphabétique des noms français, le même
    qu'ImageFolder (automobile = 0, avion = 1, ..., oiseau = 9). Mêmes tirages que charger_cifar10. Le CSV
    d'une partie est écrit en dernier : une partie dont le CSV existe est complète et n'est pas réécrite. Les
    fichiers de CIFAR-10 sont d'abord récupérés par preparer_archive_cifar10, comme dans charger_cifar10.
    """
    from pathlib import Path

    import torch
    from torchvision.datasets import CIFAR10

    noms = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]
    numeros = {nom: numero for numero, nom in enumerate(sorted(noms))}
    generateur = torch.Generator().manual_seed(seed)
    ordre = torch.randperm(50000, generator=generateur).tolist()
    parties = {"train": (True, ordre[:n_train]),
               "val": (True, ordre[n_train:n_train + n_val]),
               "test": (False, torch.randperm(10000, generator=generateur)[:n_test].tolist())}
    preparer_archive_cifar10()
    Path(dossier, "images").mkdir(parents=True, exist_ok=True)
    for partie, (entrainement, indices) in parties.items():
        fichier_csv = Path(dossier, f"{partie}.csv")
        if fichier_csv.exists():
            continue
        source = CIFAR10("data", train=entrainement, download=True)
        lignes = ["fichier,classe"]
        for numero, indice in enumerate(indices):
            image, label = source[indice]
            nom_fichier = f"{partie}_{numero:05d}.png"
            image.save(Path(dossier, "images", nom_fichier))
            lignes.append(f"{nom_fichier},{numeros[noms[label]]}")
        fichier_csv.write_text("\n".join(lignes))

In [ ]:
charger_cifar10_fichiers()

## Étape 0 : découvrir les fichiers (fournie)

Avant d'écrire le Dataset, on regarde comment les données sont rangées : le dossier d'images, puis les premières lignes du CSV d'entraînement.

In [ ]:
from pathlib import Path

dossier = Path("cifar10_fichiers")
print(len(list((dossier / "images").iterdir())), "fichiers image")
print((dossier / "train.csv").read_text().splitlines()[:4])

Les noms des classes, dans l'ordre de leurs numéros, et le nombre d'images par classe dans chaque partie.

In [ ]:
import torch

dossier = Path("cifar10_fichiers")
classes = ["automobile", "avion", "bateau", "camion", "cerf", "chat", "cheval", "chien", "grenouille", "oiseau"]
for partie in ("train", "val", "test"):
    lignes = (dossier / f"{partie}.csv").read_text().splitlines()[1:]
    effectifs = torch.bincount(torch.tensor([int(ligne.split(",")[1]) for ligne in lignes]), minlength=10)
    print(partie, len(lignes), effectifs.tolist())

Huit images d'entraînement, ouvertes directement depuis leurs fichiers, avec le nom de leur classe.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, ligne in zip(axes, (dossier / "train.csv").read_text().splitlines()[1:9]):
    fichier, classe = ligne.split(",")
    ax.imshow(Image.open(dossier / "images" / fichier))
    ax.set_title(classes[int(classe)], fontsize=9)
    ax.axis("off")
plt.show()

Questions : que contient une ligne du CSV ? Combien d'images compte chaque partie ? Les classes sont-elles équilibrées ? Quelle accuracy obtiendrait un modèle qui répond toujours la même classe ?

*Vos réponses :*

## Étape 1 : écrire le Dataset

Les transforms sont celles du TP3 (fournies) : miroir horizontal et décalage aléatoires pour l'entraînement, conversion et normalisation pour tous.

In [ ]:
import torch
from torchvision.transforms import v2

conversion = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=[0.5] * 3, std=[0.5] * 3)]
eval_tf = v2.Compose(conversion)
train_tf = v2.Compose([v2.RandomHorizontalFlip(), v2.RandomCrop(32, padding=4)] + conversion)

Complétez `ImagesCifar` : `__init__` (fourni) lit le CSV et garde la liste des couples (fichier, classe). Écrivez `__len__`, puis la lecture de l'image dans `__getitem__`.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset


class ImagesCifar(Dataset):
    def __init__(self, fichier_csv, transform=None):
        self.dossier_images = Path(fichier_csv).parent / "images"
        self.transform = transform
        lignes = Path(fichier_csv).read_text().splitlines()[1:]
        self.exemples = [ligne.split(",") for ligne in lignes]

    def __len__(self):
        # TODO : renvoyer le nombre d'exemples
        ...

    def __getitem__(self, i):
        fichier, classe = self.exemples[i]
        # TODO : ouvrir l'image self.dossier_images / fichier avec Image.open,
        #        puis lui appliquer self.transform si elle n'est pas None
        image = ...
        return image, int(classe)

Vérification (fournie) : trois Datasets, et un premier exemple.

In [ ]:
train_ds = ImagesCifar("cifar10_fichiers/train.csv", transform=train_tf)
val_ds = ImagesCifar("cifar10_fichiers/val.csv", transform=eval_tf)
test_ds = ImagesCifar("cifar10_fichiers/test.csv", transform=eval_tf)
image, label = train_ds[0]
print(len(train_ds), len(val_ds), len(test_ds), tuple(image.shape), label, classes[label])
assert len(train_ds) == 5000 and image.shape == (3, 32, 32)

## Étape 2 : les DataLoader

Créez trois DataLoader : paquets de 64 images mélangées pour l'entraînement, paquets de 256 sans mélange pour la validation et le test.

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(0)
# TODO : trois DataLoader ; batch_size=64 et shuffle=True pour l'entraînement, batch_size=256 pour les autres
train_loader = ...
val_loader = ...
test_loader = ...
images, labels = next(iter(train_loader))
print(tuple(images.shape), tuple(labels.shape), len(train_loader), "paquets par époque")
assert images.shape == (64, 3, 32, 32)

## Étape 3 : définir le CNN

Le CNN du TP3, écrit comme une classe : trois blocs `Conv2d` (kernel 3, padding 1) + `ReLU` + `MaxPool2d(2)`, avec 16, 32 puis 64 canaux, puis le classifieur `Flatten`, `Linear(64 * 4 * 4, 64)`, `ReLU`, `Linear(64, 10)`. Les deux premiers blocs sont fournis.

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class CNNCifar(nn.Module):
    def __init__(self):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        # TODO : bloc3, sur le modèle de bloc2, de 32 à 64 canaux
        self.bloc3 = ...
        # TODO : Flatten, Linear(64 * 4 * 4, 64), ReLU, Linear(64, 10)
        self.classifieur = ...

    def forward(self, x):
        # TODO : bloc1, puis bloc2, puis bloc3, puis classifieur
        ...

In [ ]:
torch.manual_seed(0)
model = CNNCifar().to(device)
assert model(images.to(device)).shape == (64, 10)
print(sum(p.numel() for p in model.parameters()), "paramètres")

## Étape 4 : entraîner

Les fonctions d'entraînement et de mesure sont fournies.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    for x, y in loader:
        loss = loss_fn(model(x.to(device)), y.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Entraînez 10 époques avec Adam (lr=1e-3), en affichant à chaque époque la loss et l'accuracy d'entraînement et de validation. Les lignes `writer` (fournies) écrivent les mêmes mesures pour TensorBoard, comme dans la démo 5 du jour 1.

In [ ]:
from torch.utils.tensorboard import SummaryWriter

loss_fn = nn.CrossEntropyLoss()
# TODO : optimiseur Adam, lr=1e-3
optimizer = ...
writer = SummaryWriter("runs/tp4")
for epoch in range(1, 11):
    # TODO : une époque d'entraînement, puis evaluate sur train_loader (train_loss, train_acc)
    #        et sur val_loader (val_loss, val_acc)
    writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
    print(f"époque {epoch}  loss {train_loss:.3f}  acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
writer.close()
assert val_acc > 0.35, "le CNN devrait dépasser 35 % en validation"

## Étape 5 : les courbes dans TensorBoard (fournie)

TensorBoard affiche les courbes écrites pendant l'entraînement : loss et accuracy, entraînement et validation. Lisez les deux courbes ensemble : s'écartent-elles ? La validation progresse-t-elle encore à la dernière époque ?

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

*Votre lecture des courbes :*

## Étape 6 : une seule mesure sur le test

Le modèle est fixé : mesurez-le une fois sur le jeu de test.

In [ ]:
# TODO : evaluate sur test_loader
loss_test, acc_test = ...
print(f"test : loss {loss_test:.3f}  accuracy {acc_test:.3f}")

## Bonus

**B1. Lire en parallèle.** Mesurez le temps d'une époque de lecture, sans modèle, avec `num_workers=0` puis `num_workers=2`.

In [ ]:
import time

for workers in (0, 2):
    # TODO : DataLoader de train_ds, batch_size=64, shuffle=True, num_workers=workers
    loader = ...
    debut = time.perf_counter()
    for paquet in loader:
        pass
    print(f"num_workers={workers} : {time.perf_counter() - debut:.2f} s pour lire une époque")

**B2. Le dernier paquet.** Avec des paquets de 128 images, combien de paquets compte une époque, avec et sans `drop_last=True` ? Pourquoi ?

In [ ]:
# TODO : len(...) de deux DataLoader de batch_size=128, sans puis avec drop_last=True